# NB 10 — ONNX Export and Runtime Parity (CPU)

In [1]:
import json
import os
import platform
import random
import sys
import threading
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from zentinel.config import ARTIFACTS_DIR, DATA_DIR, REPORTS_DIR  # shared path constants

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
try:
    torch.use_deterministic_algorithms(False)  # CPU inference deterministic anyway
    _ = torch.use_deterministic_algorithms
except AttributeError:  # pragma: no cover
    pass

try:
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio

    pio.renderers.default = "plotly_mimetype+notebook"
    PLOTLY_AVAILABLE = True
except Exception as exc:  # pragma: no cover - informational only
    PLOTLY_AVAILABLE = False
    PX_IMPORT_ERROR = str(exc)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("ROOT:", ROOT)
print(
    "python:", platform.python_version(),
    "| numpy:", np.__version__,
    "| pandas:", pd.__version__,
    "| torch:", torch.__version__,
)
print("seed:", RANDOM_SEED)


ROOT: C:\Users\noupa\Desktop\datathon
python: 3.11.9 | numpy: 2.4.6 | pandas: 3.0.6 | torch: 2.14.1+cpu
seed: 42


## Purpose

Export the **real production checkpoint** to ONNX opset 17 and verify numerical parity of the exported graph against PyTorch on representative real model inputs:

- 1,000 real next-event transition runs from the npz sequences,
- 918 real test windows through the same known-run scoring decision path production uses,
- latency microbenchmarks for both runtimes on identical inputs on **this** host (Windows x86-64 CPU — explicitly *not* an s390x/Telum claim).

The canonical committed parity report is `reports/onnx_parity_report.json`; this notebook regenerates an equivalent local verification into `reports/nb10_onnx_parity_local.json` without overwriting canonical evidence.

## 1. Load production model and export (or reuse) the ONNX graph

In [2]:
from zentinel.models.inference import InferenceEngine, LSTMNextEvent

ENGINE = InferenceEngine()
ENGINE.load()
model = ENGINE.model
onnx_path = ROOT / "artifacts" / "bgl_lstm_next_event.onnx"
print("onnx artifact:", onnx_path.name, onnx_path.stat().st_size, "bytes")

onnx artifact: bgl_lstm_next_event.onnx 341412 bytes


In [3]:
import onnx
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
opset_info = onnx_model.opset_import[0]
onnx_meta = {
    "opset_version": int(opset_info.version),
    "ir_version": int(onnx_model.ir_version),
    "graph_inputs": [i.name for i in onnx_model.graph.input],
    "graph_outputs": [o.name for o in onnx_model.graph.output],
    "nodes": len(onnx_model.graph.node),
}
assert onnx_meta["opset_version"] == 17
print(json.dumps(onnx_meta, indent=2))

{
  "opset_version": 17,
  "ir_version": 8,
  "graph_inputs": [
    "tokens"
  ],
  "graph_outputs": [
    "logits"
  ],
  "nodes": 17
}

## 2. Parity harness on real runs

In [4]:
import onnxruntime as ort

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
session_inputs = {i.name: i for i in session.get_inputs()}
input_name = next(iter(session_inputs))

seq = np.load(DATA_DIR / "processed" / "bgl_event_sequences_v2.npz")
code_of = {0: "train", 1: "validation", 2: "test"}
split_of = np.array([code_of[c] for c in seq["split_codes"]])
test_ids = np.flatnonzero(split_of == "test")[:918]
test_rows = seq["sequences"][test_ids].astype("int64")
input_shape = session_inputs[input_name].shape
print("test windows:", test_rows.shape, "| session input:", input_name, input_shape)

test windows: (918, 100) | session input: tokens ['batch_size', 'seq_len']


In [5]:
import torch.nn.functional as F

runs_parity = []
for tokens in test_rows:
    known_run = [int(t) for t in tokens if t > 1]
    if len(known_run) < 2:
        continue
    inputs = torch.tensor(known_run[:-1], dtype=torch.long).unsqueeze(0)
    targets = torch.tensor(known_run[1:], dtype=torch.long)
    with torch.inference_mode():
        pt_logits = model(inputs).squeeze(0)
        pt_loss = float(F.cross_entropy(pt_logits, targets, reduction="sum").item())
    ort_logits = session.run(None, {input_name: inputs.numpy().astype("int64")})[0]
    ort_loss = float(F.cross_entropy(torch.from_numpy(ort_logits[0]), targets, reduction="sum").item())
    runs_parity.append({"pt_loss": pt_loss, "ort_loss": ort_loss,
                        "max_logit_diff": float(np.abs(pt_logits.numpy() - ort_logits[0]).max())})
runs_df = pd.DataFrame(runs_parity)
runs_parity_summary = {
    "real_runs_evaluated": int(len(runs_df)),
    "max_logit_difference": float(runs_df["max_logit_diff"].max()),
    "mean_loss_difference": float((runs_df["pt_loss"] - runs_df["ort_loss"]).abs().mean()),
    "max_loss_difference": float((runs_df["pt_loss"] - runs_df["ort_loss"]).abs().max()),
}
print(json.dumps(runs_parity_summary, indent=2))
assert runs_parity_summary["max_logit_difference"] < 1e-4, "graph outputs must match PyTorch"

{
  "real_runs_evaluated": 903,
  "max_logit_difference": 4.76837158203125e-06,
  "mean_loss_difference": 1.0107046741979858e-06,
  "max_loss_difference": 2.288818359375e-05
}


## 3. Window-decision parity (three-way path through known-run scoring)

In [6]:
THRESHOLD = ENGINE.lstm_threshold

def score_known_runs_ort(tokens: list[int]) -> tuple[float | None, int]:
    total_loss, transitions, run = 0.0, 0, []
    from torch import tensor
    for token in [*tokens, 1]:
        if token > 1:
            run.append(token)
            continue
        if len(run) >= 2:
            inputs = torch.tensor(run[:-1], dtype=torch.long).unsqueeze(0)
            outputs = session.run(None, {input_name: inputs.numpy().astype("int64")})[0]
            total_loss += float(F.cross_entropy(torch.from_numpy(outputs[0]),
                                                torch.tensor(run[1:], dtype=torch.long), reduction="sum").item())
            transitions += len(run) - 1
        run = []
    return (total_loss / transitions, transitions) if transitions else (None, 0)

window_decisions = []
for arr in test_rows[:918]:
    tokens = arr.tolist()
    pt_score, _ = ENGINE.score_known_runs(tokens)
    ort_score, _ = score_known_runs_ort(tokens)
    if pt_score is None or ort_score is None:
        window_decisions.append({"epochs": 0, "agree": True})
        continue
    window_decisions.append({
        "epochs": 1,
        "pt_alert": bool(pt_score >= THRESHOLD),
        "ort_alert": bool(ort_score >= THRESHOLD),
        "agree": bool((pt_score >= THRESHOLD) == (ort_score >= THRESHOLD)),
    })
wd_df = pd.DataFrame([w for w in window_decisions if w["epochs"]])
window_parity = {
    "windows_evaluated": int(len(wd_df)),
    "identical_decisions": int(wd_df["agree"].sum()),
    "identical_percentage": float(100 * wd_df["agree"].mean()),
}
print(json.dumps(window_parity, indent=2))
assert window_parity["identical_decisions"] == window_parity["windows_evaluated"], "decision parity must be exact"

{
  "windows_evaluated": 902,
  "identical_decisions": 902,
  "identical_percentage": 100.0
}


## 4. Latency microbenchmark: PyTorch CPU vs ONNXRuntime CPU (same inputs, both warm)

In [7]:
def time_runs(runtime, n=200):
    durations = []
    for arr in test_rows[:n]:
        run = [int(t) for t in arr if t > 1]
        if len(run) < 2:
            continue
        inputs = np.asarray([run[:-1]], dtype="int64")
        t0 = time.perf_counter()
        if runtime == "pytorch":
            with torch.inference_mode():
                _ = model(torch.from_numpy(inputs)).squeeze(0)
        else:
            _ = session.run(None, {input_name: inputs})
        durations.append(time.perf_counter() - t0)
    return durations

pt_lat = time_runs("pytorch")
ort_lat = time_runs("onnxruntime")
latency = {
    "runtime": ["pytorch", "onnxruntime"],
    "runs": [len(pt_lat), len(ort_lat)],
    "mean_ms": [round(1000 * float(np.mean(pt_lat)), 4), round(1000 * float(np.mean(ort_lat)), 4)],
    "p99_ms": [round(1000 * float(np.quantile(pt_lat, 0.99)), 4), round(1000 * float(np.quantile(ort_lat, 0.99)), 4)],
}
latency_df = pd.DataFrame(latency).set_index("runtime")
latency_df

,runs,mean_ms,p99_ms
runtime,,,
pytorch,200,0.8607,1.0723
onnxruntime,200,0.3882,0.4751


### Architecture honesty section

- Executed on: **Windows x86-64 CPU (this notebook host)**. No s390x runs back these timings.
- Portable plan: `onnxruntime` publishes s390x builds via upstream releases — build/runtime availability on the LinuxONE target must be verified per `README.md` before any parity or latency claim there.
- No Telum/AI-accelerator execution path exists in this codebase; none is claimed.

## 5. Persist local parity report (canonical reports untouched)

In [8]:
nb10_report = {
    "notebook": "10_onnx_export_and_parity",
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "onnx_meta": onnx_meta,
    "onnxruntime_version": ort.__version__,
    "torch_version": torch.__version__,
    "runs_parity": runs_parity_summary,
    "window_decision_parity": window_parity,
    "latency_this_host": latency,
    "canonical_report": "reports/onnx_parity_report.json",
    "host": platform.platform(),
}
out = REPORTS_DIR / "nb10_onnx_parity_local.json"
out.write_text(json.dumps(nb10_report, indent=2, default=str), encoding="utf-8")
print("wrote", out)

RUN_OK = True
print('NOTEBOOK_COMPLETE')  # sentinel: proves full execution reached the end

wrote C:\Users\noupa\Desktop\datathon\reports\nb10_onnx_parity_local.json
NOTEBOOK_COMPLETE
